In [18]:
# Preliminary features & parameters
import src.simulations.backtest_portfolio_v2 as bp
from src import indicators

features = {
        'sma_trend_regime': {
            'func': indicators.calculate_sma_crossover,
            'params': {'fast_window': 10, 'slow_window': 30, 'binary': True},
        },
        'sma_position': {
            'func': indicators.calculate_sma_position,
            'params': {'window': 25},
        },
        'rsi': {
            'func': indicators.calculate_rsi,
            'params': {'window': 14},
        },
        'bollinger_position': {
            'func': indicators.calculate_bollinger_position,
            'params': {'window': 20, 'num_std': 2},
        },
        'price_roc': {
            'func': indicators.calculate_roc,
            'params': {'window': 5}
        }
    }

ticker_pool = [
        "NVDA", "MSFT", "AVGO", "NOW",
        "ORCL", "AAPL", "TEAM", "INTC",
        "SNOW", "WIX",  "AMD",  "CSCO",
        "SHOP", "AMZN", "CRM",
    ]
 
benchmark = "SPY"

In [19]:
# Functionize model testing
from sklearn import ensemble
import src.stock_screener as sc

def run_with_model(input_model, features_dict):
    engine = bp.PortfolioBacktestEngine(
        model=input_model,
        feature_configs=features_dict,
        confidence_threshold=0.75,
        stop_loss=0.05,
        min_hold_days=2,
        adx_threshold=20,
        training_years=2,
        testing_years=1,
        offset_years=0
    )

    total_years = engine.training_years + engine.testing_years + engine.offset_years
    print(f"Downloading {total_years}y of data for {len(ticker_pool)} tickers + {benchmark}...")
    master_df = sc.fetch_screener_data(
        ticker_pool + [benchmark], period=f"{total_years}y", interval="1d"
    )
    print("Download Complete.")

    benchmark_prices = master_df['Close'][benchmark]
    benchmark_prices.name = benchmark

    engine.run_simulation(
        ticker_pool=ticker_pool,
        pre_downloaded_df=master_df,
        benchmark_prices=benchmark_prices,
    )

    print("\nTrade Log:")
    print(engine.last_trade_log.to_string(index=False))

In [20]:
rf_classifier = ensemble.RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        min_samples_split=10,
        max_features='sqrt',
        max_depth=10
    )
run_with_model(input_model=rf_classifier, features_dict=features)

Fetching historical data for companies: ['NVDA', 'MSFT', 'AVGO', 'NOW', 'ORCL', 'AAPL', 'TEAM', 'INTC', 'SNOW', 'WIX', 'AMD', 'CSCO', 'SHOP', 'AMZN', 'CRM', 'SPY']...


[*********************100%***********************]  16 of 16 completed


Download Complete.

Portfolio Simulation Results
  Train window : 2023-06-20 to 2025-06-18
  Test window  : 2025-06-18 to 2026-06-18
  Strategy Return   : +29.59%
  Benchmark (SPY) : +19.01%
  Alpha             : +10.58%
  Win Rate          : 57.75%
  Total Trades      : 71

Trade Log:
     type ticker      price       date     return
      BUY    WIX 152.369995 2025-07-24        NaN
     SELL    WIX 150.500000 2025-07-28  -1.227273
      BUY   INTC  20.680000 2025-07-28        NaN
     SELL   INTC  19.799999 2025-07-31  -4.255324
      BUY    NOW 188.623993 2025-07-31        NaN
STOP_LOSS    NOW 174.824005 2025-08-07  -7.316136
      BUY    NOW 174.824005 2025-08-07        NaN
     SELL    NOW 178.410004 2025-08-18   2.051205
      BUY   CSCO  65.837204 2025-08-18        NaN
     SELL   CSCO  65.994537 2025-08-20   0.238973
      BUY   AVGO 289.509583 2025-08-20        NaN
     SELL   AVGO 292.323425 2025-08-22   0.971934
      BUY   ORCL 234.519348 2025-08-22        NaN
     SELL   O

In [23]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05)

run_with_model(input_model=xgb_model, features_dict=features)

Fetching historical data for companies: ['NVDA', 'MSFT', 'AVGO', 'NOW', 'ORCL', 'AAPL', 'TEAM', 'INTC', 'SNOW', 'WIX', 'AMD', 'CSCO', 'SHOP', 'AMZN', 'CRM', 'SPY']...


[*********************100%***********************]  16 of 16 completed


Download Complete.

Portfolio Simulation Results
  Train window : 2023-06-20 to 2025-06-18
  Test window  : 2025-06-18 to 2026-06-18
  Strategy Return   : -17.06%
  Benchmark (SPY) : +19.01%
  Alpha             : -36.07%
  Win Rate          : 52.70%
  Total Trades      : 74

Trade Log:
     type ticker      price       date    return
      BUY    WIX 152.369995 2025-07-24       NaN
     SELL    WIX 150.500000 2025-07-28 -1.227273
      BUY   INTC  20.680000 2025-07-28       NaN
STOP_LOSS   INTC  19.309999 2025-08-01 -6.624762
      BUY    NOW 182.873993 2025-08-01       NaN
     SELL    NOW 174.501999 2025-08-08 -4.578012
      BUY   TEAM 168.059998 2025-08-08       NaN
STOP_LOSS   TEAM 159.279999 2025-08-11 -5.224324
      BUY    NOW 171.274002 2025-08-11       NaN
     SELL    NOW 170.171997 2025-08-14 -0.643416
      BUY   SNOW 194.899994 2025-08-14       NaN
     SELL   SNOW 198.240005 2025-08-18  1.713705
      BUY   CSCO  65.837204 2025-08-18       NaN
     SELL   CSCO  65.994545

In [ ]:
from sklearn.linear_model import LogisticRegression

# There was some noise based on dividing by 0, etc. that we filter out
import warnings
warnings.filterwarnings('ignore')

logreg_model = LogisticRegression(C=0.1)

run_with_model(input_model=logreg_model, features_dict=features)

Fetching historical data for companies: ['NVDA', 'MSFT', 'AVGO', 'NOW', 'ORCL', 'AAPL', 'TEAM', 'INTC', 'SNOW', 'WIX', 'AMD', 'CSCO', 'SHOP', 'AMZN', 'CRM', 'SPY']...


[*********************100%***********************]  16 of 16 completed


Download Complete.

Portfolio Simulation Results
  Train window : 2023-06-20 to 2025-06-18
  Test window  : 2025-06-18 to 2026-06-18
  Strategy Return   : -1.38%
  Benchmark (SPY) : +19.01%
  Alpha             : -20.39%
  Win Rate          : 53.16%
  Total Trades      : 79

Trade Log:
     type ticker      price       date     return
      BUY    WIX 152.369995 2025-07-24        NaN
     SELL    WIX 150.500000 2025-07-28  -1.227273
      BUY   INTC  20.680000 2025-07-28        NaN
     SELL   INTC  20.340000 2025-07-30  -1.644101
      BUY    WIX 139.369995 2025-07-30        NaN
STOP_LOSS    WIX 128.970001 2025-08-01  -7.462147
      BUY    WIX 128.970001 2025-08-01        NaN
     SELL    WIX 133.490005 2025-08-07   3.504694
      BUY   TEAM 171.000000 2025-08-07        NaN
STOP_LOSS   TEAM 159.279999 2025-08-11  -6.853802
      BUY   TEAM 159.279999 2025-08-11        NaN
     SELL   TEAM 164.389999 2025-08-14   3.208187
      BUY    WIX 120.389999 2025-08-14        NaN
     SELL    W